# BERT-QPP$_{cross}$ on TREC DL 2019 / 2020

Runs a trained BERT-QPP$_{cross}$ checkpoint over pre-existing run files (already on Drive, no live retrieval) for 8 rankers, and correlates predicted QPP scores against actual MAP@50, nDCG@100, and nDCG@10 (via `pytrec_eval`) for:
- TREC DL 2019 alone
- TREC DL 2020 alone
- TREC DL 2019 + 2020 pooled

For each scope/metric, four correlation views are reported (Pearson + Kendall only, no Spearman):
- **per_system** — correlation within each ranker's own queries (Single Ranker, Multiple Queries).
- **ranker_macro_avg (SRMQ)** — mean of each ranker's own `per_system` correlation, averaged across the 8 rankers.
- **overall (MRMQ)** — every `(system, query)` pair in the scope pooled into one correlation (Multiple Rankers, Multiple Queries).
- **per_query_macro_avg (MRSQ)** — for each query, correlate predicted vs. actual *across the 8 rankers* (Multiple Rankers, Single Query), then average across queries.

No Pyserini/PyTerrier and no live indexing — run files, queries, and qrels are read directly from Drive. The trained checkpoint is fetched via `gdown` on first run and cached on Drive; subsequent runs reuse the cached copy. The only other external fetch is the MS MARCO passage collection (needed for each top-ranked document's text), also cached on Drive after the first run.

Sanity-check cells are interspersed after each major stage (loading, doc-text resolution, prediction, ground truth, correlation) to catch data problems where they happen rather than as a confusing symptom several cells later.

## 1. Mount Drive & install dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q sentence-transformers pytrec_eval scipy pandas tqdm gdown

## 2. Configuration — edit paths to match your Drive layout

In [ ]:
import os

drive_base_runfiles_19 = "/content/drive/MyDrive/precise-qpp/data/TRECDL19-runfiles"
drive_base_runfiles_20 = "/content/drive/MyDrive/precise-qpp/data/TRECDL20-runfiles"
drive_base_queries_qrels = "/content/drive/MyDrive/precise-qpp/data/TREC-queries-qrels"

# Trained checkpoint: downloaded via gdown into this Drive path on first run, reused on every run after.
MODEL_DRIVE_FOLDER_URL = "https://drive.google.com/drive/folders/1NDZzEpaay0cDumTKDUSMmv99sg9FyHrL"
MODEL_PATH = "/content/drive/MyDrive/precise-qpp/models/tuned_model-ce_bert-base-uncased_e1_b8"

# MS MARCO passage collection, needed for doc text lookup. Downloaded once and cached here if missing.
COLLECTION_PATH = "/content/drive/MyDrive/precise-qpp/data/collection.tsv"  # edit me if you already have it cached elsewhere

OUT_DIR = "/content/drive/MyDrive/precise-qpp/results/bertqpp_cross_dl1920"
os.makedirs(OUT_DIR, exist_ok=True)

SYSTEM_NAMES = ["BM25", "rm3", "colbert.e2e", "e5", "monot5", "prf_rank_beta05", "splade", "prf_rerank_beta05"]

def run_files_for_year(base_dir, year):
    """Build the fixed list of per-ranker run files for TREC DL `year` ('2019' or '2020')."""
    yy = year[-2:]
    return [
        f"{base_dir}/BM25.{year}.100.res",
        f"{base_dir}/rm3.100.res",
        f"{base_dir}/colbert.e2e.100.res",
        f"{base_dir}/e5_dl_{yy}.100.res",
        f"{base_dir}/monot5.100.res",
        f"{base_dir}/prf_rank_beta05.{year}.100.res",
        f"{base_dir}/splade.100.res",
        f"{base_dir}/prf_rerank_beta05.{year}.100.res",
    ]

def qrels_path(year):
    return f"{drive_base_queries_qrels}/pass_{year}.qrels"

def queries_path(year):
    return f"{drive_base_queries_qrels}/pass_{year}.queries"

YEAR_RUN_DIRS = {"2019": drive_base_runfiles_19, "2020": drive_base_runfiles_20}

## 3. Parsers for queries / qrels / TREC run files

In [ ]:
import pytrec_eval
from collections import defaultdict

def parse_queries(path):
    queries = {}
    with open(path) as f:
        for line in f:
            line = line.rstrip('\n')
            if not line:
                continue
            qid, text = line.split('\t', 1)
            queries[qid] = text
    return queries

def parse_qrels(path):
    with open(path) as f:
        return pytrec_eval.parse_qrel(f)

def parse_run(path):
    """TREC 6-col run file -> (run_scores: {qid: {docid: score}}, top1_doc: {qid: docid}).
    top1 is picked by lowest rank column, not by file order, so it's robust to unsorted runs."""
    run_scores = defaultdict(dict)
    best_rank = {}
    top1 = {}
    with open(path) as f:
        for line in f:
            parts = line.split()
            if len(parts) < 6:
                continue
            qid, _, docid, rank, score, _ = parts[:6]
            run_scores[qid][docid] = float(score)
            rank = int(rank)
            if qid not in best_rank or rank < best_rank[qid]:
                best_rank[qid] = rank
                top1[qid] = docid
    return dict(run_scores), top1

## 4. Load queries, qrels, and every system's run file for both years

In [ ]:
year_queries = {}
year_qrels = {}
run_scores_by_system = {}   # (year, system) -> {qid: {docid: score}}
top1_by_system = {}         # (year, system) -> {qid: docid}
needed_docids = set()

for year, base_dir in YEAR_RUN_DIRS.items():
    year_queries[year] = parse_queries(queries_path(year))
    year_qrels[year] = parse_qrels(qrels_path(year))
    print(f"DL{year[-2:]}: {len(year_queries[year])} queries, {len(year_qrels[year])} judged")

    for system, run_file in zip(SYSTEM_NAMES, run_files_for_year(base_dir, year)):
        if not os.path.exists(run_file) or os.path.getsize(run_file) == 0:
            print(f"  [SKIP] {system}: missing or empty ({run_file})")
            continue
        run_scores, top1 = parse_run(run_file)
        run_scores_by_system[(year, system)] = run_scores
        top1_by_system[(year, system)] = top1
        needed_docids.update(top1.values())
        print(f"  [OK]   {system}: {len(run_scores)} queries")

print(f"\nNeed text for {len(needed_docids)} unique top-1 docids across both years/systems")

In [ ]:
# --- Sanity check: query/qrels/run loading ---
assert year_queries["2019"] and year_queries["2020"], "Queries failed to load for one or both years"
assert year_qrels["2019"] and year_qrels["2020"], "Qrels failed to load for one or both years"

overlap = set(year_queries["2019"]) & set(year_queries["2020"])
print(f"Query id overlap between DL19 and DL20: {len(overlap)} (must be 0 for DL19+20 pooling to be valid)")
assert not overlap, f"DL19/DL20 share query ids -- pooling would double-count: {sorted(overlap)[:10]}"

expected_pairs = len(YEAR_RUN_DIRS) * len(SYSTEM_NAMES)
print(f"(year, system) pairs loaded: {len(run_scores_by_system)} / {expected_pairs} possible")

for key, top1 in top1_by_system.items():
    assert len(top1) > 0, f"{key}: top1 dict is empty"
    assert len(top1) == len(run_scores_by_system[key]), (
        f"{key}: top1 has {len(top1)} queries but run_scores has {len(run_scores_by_system[key])}"
    )
print("[OK] every loaded system has a non-empty top1 doc for each of its scored queries")

## 5. Resolve top-1 document text
Downloads and caches the MS MARCO collection on Drive once, then does a single sequential scan pulling out only the docids actually needed (a few hundred, not all 8.8M passages) — keeps memory use small.

In [ ]:
import subprocess

# msmarco.blob.core.windows.net now returns 409 (public access disabled on that storage
# account). z22.web.core.windows.net is the same dataset's static-website endpoint, still
# public; Dropbox is a community mirror. Both are verified against the known MD5 before use.
MSMARCO_URLS = [
    "https://msmarco.z22.web.core.windows.net/msmarcoranking/collectionandqueries.tar.gz",
    "https://www.dropbox.com/s/9f54jg2f71ray3b/collectionandqueries.tar.gz?dl=1",
]
MSMARCO_MD5 = "31644046b18952c1386cd4564ba2ae69"

if not os.path.exists(COLLECTION_PATH):
    os.makedirs(os.path.dirname(COLLECTION_PATH), exist_ok=True)
    archive = "/content/collectionandqueries.tar.gz"

    verified = False
    for url in MSMARCO_URLS:
        print(f"[INFO] Trying {url}")
        subprocess.run(["rm", "-f", archive])
        dl = subprocess.run(["bash", "-c", f'wget -q --show-progress -O "{archive}" "{url}"'])
        if dl.returncode != 0:
            print(f"[WARN] Download failed from {url}, trying next source")
            continue
        md5 = subprocess.run(["md5sum", archive], capture_output=True, text=True).stdout.split()[0]
        if md5 != MSMARCO_MD5:
            print(f"[WARN] MD5 mismatch from {url} (got {md5}, expected {MSMARCO_MD5}), trying next source")
            continue
        verified = True
        break

    if not verified:
        raise RuntimeError(
            f"Could not download a valid collectionandqueries.tar.gz from any known source ({MSMARCO_URLS}). "
            f"Download it manually, extract collection.tsv, and place it at {COLLECTION_PATH} on Drive -- "
            "this cell detects the cached file and skips the download on the next run."
        )

    extract = subprocess.run(["bash", "-c",
        f'tar -xzf "{archive}" -C /content collection.tsv && mv /content/collection.tsv "{COLLECTION_PATH}" && rm "{archive}"'])
    if extract.returncode != 0:
        raise RuntimeError("Archive was verified but extracting collection.tsv from it failed.")
else:
    print(f"[INFO] Using cached collection at {COLLECTION_PATH}")

doc_text = {}
with open(COLLECTION_PATH) as f:
    for line in f:
        docid, text = line.rstrip('\n').split('\t', 1)
        if docid in needed_docids:
            doc_text[docid] = text
            if len(doc_text) == len(needed_docids):
                break

missing = needed_docids - doc_text.keys()
print(f"Resolved {len(doc_text)}/{len(needed_docids)} docids ({len(missing)} missing)")

In [ ]:
# --- Sanity check: doc text resolution ---
assert doc_text, "doc_text is empty -- collection lookup produced nothing"

sample_docids = list(doc_text.keys())[:3]
print("Spot-check resolved passages:")
for docid in sample_docids:
    text = doc_text[docid]
    assert isinstance(text, str) and text.strip(), f"docid {docid} resolved to empty/invalid text"
    print(f"  {docid}: {text[:120]}{'...' if len(text) > 120 else ''}")

print("\nPer-system doc-text coverage:")
for key, top1 in top1_by_system.items():
    resolved = sum(1 for d in top1.values() if d in doc_text)
    pct = 100 * resolved / len(top1)
    flag = "  [WARN] low coverage" if pct < 90 else ""
    print(f"  {key}: {resolved}/{len(top1)} ({pct:.0f}%){flag}")

## 6. Fetch trained model (cached on Drive) & predict QPP scores
First run downloads the checkpoint via `gdown` into `MODEL_PATH` on Drive; every run after reuses that cached copy — no repeat downloads.

In [ ]:
if not os.path.isdir(MODEL_PATH) or not os.listdir(MODEL_PATH):
    os.makedirs(MODEL_PATH, exist_ok=True)
    !gdown --folder "$MODEL_DRIVE_FOLDER_URL" -O "$MODEL_PATH"
else:
    print(f"[INFO] Using cached model at {MODEL_PATH}")

In [ ]:
from sentence_transformers.cross_encoder import CrossEncoder
from tqdm import tqdm

model = CrossEncoder(MODEL_PATH, num_labels=1)

predictions = {}  # (year, system) -> {qid: predicted_score}
for (year, system), top1 in tqdm(top1_by_system.items()):
    qids = [qid for qid in top1 if top1[qid] in doc_text]
    pairs = [[year_queries[year][qid], doc_text[top1[qid]]] for qid in qids]
    scores = model.predict(pairs, show_progress_bar=False)
    predictions[(year, system)] = {qid: float(s) for qid, s in zip(qids, scores)}

In [ ]:
# --- Sanity check: predicted QPP scores ---
import numpy as np

print("Per-system prediction range (min / max / mean / std):")
for key, preds in predictions.items():
    vals = np.array(list(preds.values()))
    coverage_gap = len(top1_by_system[key]) - len(preds)
    flags = []
    if np.isnan(vals).any():
        flags.append("contains NaN")
    if vals.std() < 1e-6:
        flags.append("near-constant predictions")
    if coverage_gap > 0:
        flags.append(f"{coverage_gap} queries dropped (doc text missing)")
    flag_str = f"  [WARN] {', '.join(flags)}" if flags else ""
    print(f"  {key}: n={len(vals)} min={vals.min():.4f} max={vals.max():.4f} "
          f"mean={vals.mean():.4f} std={vals.std():.4f}{flag_str}")

## 7. Actual per-query effectiveness (MAP@50, nDCG@100, nDCG@10) via `pytrec_eval`

In [ ]:
METRICS = {"map_cut.50", "ndcg_cut.100", "ndcg_cut.10"}
METRIC_KEYS = {"MAP@50": "map_cut_50", "nDCG@100": "ndcg_cut_100", "nDCG@10": "ndcg_cut_10"}

actual_scores_by_system = {}  # (year, system) -> {qid: {metric_key: value}}
for (year, system), run_scores in run_scores_by_system.items():
    evaluator = pytrec_eval.RelevanceEvaluator(year_qrels[year], METRICS)
    actual_scores_by_system[(year, system)] = evaluator.evaluate(run_scores)

In [ ]:
# --- Sanity check: ground-truth metric values ---
print("Per-system ground-truth coverage & value-range check (MAP/nDCG must lie in [0, 1]):")
for key, scores in actual_scores_by_system.items():
    assert scores, f"{key}: pytrec_eval returned no per-query scores"
    for metric_label, metric_key in METRIC_KEYS.items():
        vals = [v[metric_key] for v in scores.values() if metric_key in v]
        assert vals, f"{key}: no queries have {metric_label} computed"
        lo, hi = min(vals), max(vals)
        assert -1e-9 <= lo and hi <= 1 + 1e-9, f"{key} {metric_label}: value out of [0,1] range ({lo}, {hi})"
    print(f"  {key}: {len(scores)} queries scored")
print("[OK] every system has all 3 metrics computed, all values within [0, 1]")

## 8. Correlate predicted vs. actual — per-system, ranker macro-avg, overall, and per-query macro-average
For DL19, DL20, and DL19+20 pooled. Pearson and Kendall only.

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, kendalltau

SCOPES = {"DL19": ["2019"], "DL20": ["2020"], "DL19+20": ["2019", "2020"]}

rows = []
for scope, years in SCOPES.items():
    # Pool predictions/actuals across years-in-scope, per system.
    system_pred, system_actual = {}, {}
    for system in SYSTEM_NAMES:
        pred, actual = {}, {}
        for year in years:
            key = (year, system)
            if key not in predictions:
                continue
            pred.update(predictions[key])
            actual.update(actual_scores_by_system[key])
        if pred:
            system_pred[system] = pred
            system_actual[system] = actual

    for metric_label, metric_key in METRIC_KEYS.items():
        # --- per_system: correlation within each ranker's own queries (Single Ranker, Multiple Queries) ---
        system_pearsons, system_kendalls = [], []
        for system, pred in system_pred.items():
            actual = system_actual[system]
            common = [qid for qid in pred if qid in actual]
            if len(common) < 2:
                continue
            p = [pred[qid] for qid in common]
            a = [actual[qid][metric_key] for qid in common]
            pr, kt = pearsonr(p, a)[0], kendalltau(p, a)[0]
            rows.append({"scope": scope, "metric": metric_label, "level": "per_system", "system": system,
                         "n": len(common), "pearson": pr, "kendall": kt})
            if not (np.isnan(pr) or np.isnan(kt)):
                system_pearsons.append(pr)
                system_kendalls.append(kt)

        # --- ranker_macro_avg (SRMQ): mean of each ranker's own correlation, averaged across rankers ---
        if system_pearsons:
            rows.append({"scope": scope, "metric": metric_label, "level": "ranker_macro_avg", "system": "ALL",
                         "n": len(system_pearsons), "pearson": float(np.mean(system_pearsons)), "kendall": float(np.mean(system_kendalls))})

        # --- overall (MRMQ): every (system, query) pair in the scope pooled together ---
        all_p, all_a = [], []
        for system, pred in system_pred.items():
            actual = system_actual[system]
            for qid in pred:
                if qid in actual:
                    all_p.append(pred[qid])
                    all_a.append(actual[qid][metric_key])
        if len(all_p) >= 2:
            rows.append({"scope": scope, "metric": metric_label, "level": "overall", "system": "ALL",
                         "n": len(all_p), "pearson": pearsonr(all_p, all_a)[0], "kendall": kendalltau(all_p, all_a)[0]})

        # --- per_query_macro_avg (MRSQ): correlate across systems for each query, then average across queries ---
        qids = set()
        for pred in system_pred.values():
            qids.update(pred.keys())

        q_pearsons, q_kendalls = [], []
        for qid in qids:
            p = [system_pred[s][qid] for s in system_pred if qid in system_pred[s] and qid in system_actual[s]]
            a = [system_actual[s][qid][metric_key] for s in system_pred if qid in system_pred[s] and qid in system_actual[s]]
            if len(p) < 2:
                continue
            pr, kt = pearsonr(p, a)[0], kendalltau(p, a)[0]
            if not (np.isnan(pr) or np.isnan(kt)):
                q_pearsons.append(pr)
                q_kendalls.append(kt)

        if q_pearsons:
            rows.append({"scope": scope, "metric": metric_label, "level": "per_query_macro_avg", "system": "ALL",
                         "n": len(q_pearsons), "pearson": float(np.mean(q_pearsons)), "kendall": float(np.mean(q_kendalls))})

results_df = pd.DataFrame(rows)
results_df

In [ ]:
# --- Sanity check: cross-validate n counts in results_df ---
nan_rows = results_df[results_df[["pearson", "kendall"]].isna().any(axis=1)]
if len(nan_rows):
    print(f"[WARN] {len(nan_rows)} rows have NaN pearson/kendall:")
    display(nan_rows)
else:
    print("[OK] no NaN pearson/kendall values in results_df")

def get_n(level, scope, system, metric_label):
    row = results_df[(results_df.level == level) & (results_df.scope == scope)
                      & (results_df.system == system) & (results_df.metric == metric_label)]
    return row["n"].iloc[0] if len(row) else None

# DL19+20's per-system query count should equal DL19's + DL20's (years must not overlap)
mismatches = 0
for system in SYSTEM_NAMES:
    for metric_label in METRIC_KEYS:
        n19 = get_n("per_system", "DL19", system, metric_label)
        n20 = get_n("per_system", "DL20", system, metric_label)
        n1920 = get_n("per_system", "DL19+20", system, metric_label)
        if n19 is not None and n20 is not None and n1920 != n19 + n20:
            mismatches += 1
            print(f"  [WARN] {system}/{metric_label}: DL19+20 n={n1920} != DL19 n={n19} + DL20 n={n20}")
print("[OK] DL19+20 per-system query counts equal DL19 + DL20 for all systems/metrics"
      if mismatches == 0 else f"[WARN] {mismatches} scope-pooling mismatches found above")

# overall's pooled pair count should equal the sum of every system's per_system n, for the same scope/metric
mismatches = 0
for scope in SCOPES:
    for metric_label in METRIC_KEYS:
        per_system_n = results_df[(results_df.level == "per_system") & (results_df.scope == scope)
                                   & (results_df.metric == metric_label)]["n"].sum()
        overall_n = get_n("overall", scope, "ALL", metric_label)
        if overall_n is not None and overall_n != per_system_n:
            mismatches += 1
            print(f"  [WARN] {scope}/{metric_label}: overall n={overall_n} != sum of per_system n={per_system_n}")
print("[OK] overall n matches sum of per-system n for every scope/metric"
      if mismatches == 0 else f"[WARN] {mismatches} overall-vs-per-system mismatches found above")

## 9. Save results

In [ ]:
csv_path = f"{OUT_DIR}/correlations.csv"
results_df.to_csv(csv_path, index=False)
print(f"Saved correlation table to {csv_path}")

print("\n--- Per-system (Pearson) ---")
display(results_df[results_df.level == "per_system"].pivot_table(index=["system", "scope"], columns="metric", values="pearson"))

print("\n--- Ranker macro-avg, overall, & per-query macro-average (Pearson) ---")
display(results_df[results_df.level != "per_system"].pivot_table(index=["level", "scope"], columns="metric", values="pearson"))

In [ ]:
LEVEL_LABELS = {"ranker_macro_avg": "SRMQ", "per_query_macro_avg": "MRSQ", "overall": "MRMQ"}

agg_df = results_df[results_df.level.isin(LEVEL_LABELS)].copy()
agg_df["level"] = agg_df["level"].map(LEVEL_LABELS)

kendall_table = agg_df.pivot_table(index="scope", columns=["metric", "level"], values="kendall")
column_order = pd.MultiIndex.from_product([METRIC_KEYS.keys(), ["SRMQ", "MRSQ", "MRMQ"]])
kendall_table = kendall_table.reindex(columns=column_order)
kendall_table = kendall_table.reindex(["DL19", "DL20", "DL19+20"])

print("--- Kendall's tau: SRMQ (single-ranker/multi-query, ranker-macro-avg), "
      "MRSQ (multi-ranker/single-query, per-query-macro-avg), MRMQ (multi-ranker/multi-query, overall) ---")
display(kendall_table)